In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
# Conda environment ir-class with numpy, pandas, scikit-learn, joblib.
"""
conda activate ir-class
"""


In [ ]:
# === Cell 3: Imports ===

import ast
import os
import re
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline        import Pipeline
from sklearn.decomposition   import PCA
from sklearn.svm             import SVC
from sklearn.metrics         import (
    f1_score, accuracy_score, classification_report,
    roc_auc_score,
)
from sklearn.preprocessing   import StandardScaler, label_binarize
import joblib

from IPython.display import display


In [ ]:
# === Cell 4: Config ===

# --- HPO results from the classifier benchmark (frozen hyperparameters) ---
HPO_RESULTS_CSV = Path(
    f"{KI_SORT_ROOT}/KI_Sort_Project/05_IR_Classification"
    "/20260629_Classifier_Benchmark_Best_Preprocessing/results"
    "/20260629_Classifier_Benchmark_Best_Preprocessing_hpo_results.csv"
)
SELECTED_CLASSIFIER = "SVM"

# --- Data sources (same as the classifier benchmark) ---
SPECTRA_DIR      = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/01_Preprocessing_Investigation/C2_als_z-score")
GROUND_TRUTH_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/spectra_ground_truth_v7.csv")

# Deployment training: the cv/test split is ignored and all spectra are used.

# --- Output paths ---
DEPLOYMENT_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/05_IR_Classification/20260629_SVM_Deployment_Training")
RESULTS_DIR     = DEPLOYMENT_ROOT / "results"
MODELS_DIR      = RESULTS_DIR / "models"
for p in [DEPLOYMENT_ROOT, RESULTS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_NAME = DEPLOYMENT_ROOT.name
RANDOM_SEED  = 42

# --- Dataset semantics (same as the classifier benchmark) ---
FILENAME_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_id>\d+)_(?P<region>body|cap|label)$",
    re.IGNORECASE,
)
OBJECT_ID_WIDTH = 2
VALID_REGIONS   = ["body", "cap", "label"]

REGION_TO_GT_COLUMN = {
    "body":  "body_class",
    "label": "label_class",
    "cap":   "cap_class",
}

# --- Output artefact paths ---
DEPLOYMENT_MODEL_JOBLIB = MODELS_DIR / f"{PROJECT_NAME}_SVM_deployment.joblib"
TRAINING_SUMMARY_CSV    = RESULTS_DIR / f"{PROJECT_NAME}_training_summary.csv"

print(f"Project:              {PROJECT_NAME}")
print(f"Selected classifier:  {SELECTED_CLASSIFIER}")
print(f"HPO source:           {HPO_RESULTS_CSV.name}")


In [ ]:
# === Cell 5: Environment and Path Check ===

import sklearn
print("Python:       ", sys.version.split()[0])
print("numpy:        ", np.__version__)
print("pandas:       ", pd.__version__)
print("scikit-learn: ", sklearn.__version__)
print("Platform:     ", platform.platform())
print()

all_ok = True
for name, path in [
    ("HPO_RESULTS_CSV", HPO_RESULTS_CSV),
    ("SPECTRA_DIR",     SPECTRA_DIR),
    ("GROUND_TRUTH_CSV", GROUND_TRUTH_CSV),
]:
    status = "OK     " if path.exists() else "MISSING"
    print(f"  {status}  {name}: {path}")
    if not path.exists():
        all_ok = False


def read_spectrum_file(path):
    """Load a semicolon-delimited wavenumber;reflectance file, sorted by wavenumber."""
    df = pd.read_csv(path, sep=";", header=None,
                     names=["wavenumber", "reflectance"], engine="python")
    if df.shape[1] != 2:
        raise ValueError(f"Expected 2 columns, got {df.shape[1]}: {path}")
    df["wavenumber"]  = pd.to_numeric(df["wavenumber"],  errors="coerce")
    df["reflectance"] = pd.to_numeric(df["reflectance"], errors="coerce")
    if df.isna().any().any():
        raise ValueError(f"Non-numeric values in {path}")
    if df.empty:
        raise ValueError(f"Empty file: {path}")
    return df.sort_values("wavenumber").reset_index(drop=True)


if not all_ok:
    raise RuntimeError("Sanity check FAILED — fix paths before continuing.")
print("\nSanity check PASS")


In [ ]:
# === Cell 6: File Discovery and Parsing ===

def parse_filename(path):
    """Extract material, object_id, region and object_stem from a filename; None if no match."""
    match = FILENAME_REGEX.match(Path(path).stem)
    if not match:
        return None
    d   = match.groupdict()
    mat = d["material"].upper()
    oid = d["object_id"].zfill(OBJECT_ID_WIDTH)
    stem = f"{mat}_obj_{oid}"
    return {
        "path":            str(path),
        "filename":        Path(path).name,
        "prefix_material": mat,
        "object_id":       oid,
        "region":          d["region"].lower(),
        "object_stem":     stem,
        "group_id":        stem,
    }


all_files     = sorted([p for p in SPECTRA_DIR.iterdir() if p.is_file()])
parsed_rows   = []
skipped_files = []
for p in all_files:
    res = parse_filename(p)
    (parsed_rows if res is not None else skipped_files).append(
        res if res is not None else p.name
    )

if skipped_files:
    print(f"WARNING: {len(skipped_files)} files did not match FILENAME_REGEX and are excluded:")
    for name in skipped_files[:10]:
        print(f"  - {name}")

raw_meta_df = pd.DataFrame(parsed_rows)
if raw_meta_df.empty:
    raise RuntimeError(f"No files matched FILENAME_REGEX in {SPECTRA_DIR}.")

print(f"Files discovered:  {len(all_files)}")
print(f"Files parsed:      {len(raw_meta_df)}")
print(f"Unique objects:    {raw_meta_df['group_id'].nunique()}")
print(f"Unique regions:    {sorted(raw_meta_df['region'].unique())}")


In [ ]:
# === Cell 7: Ground Truth Assignment ===

# Deployment training: all labelled spectra are used (no cv/test split).

# (a) Load ground truth
gt_df = pd.read_csv(GROUND_TRUTH_CSV, sep=";", dtype=str)
gt_df.columns = [c.strip() for c in gt_df.columns]
for c in gt_df.columns:
    gt_df[c] = gt_df[c].str.strip().replace({"": np.nan})

print("Ground-truth columns:", list(gt_df.columns))
print(f"Ground-truth objects: {len(gt_df)}")
gt_indexed = gt_df.set_index("filename")


# (b) Assign region-specific true class
def lookup_true_class(row):
    """Return the ground-truth polymer class for this spectrum's region, or NaN."""
    stem = row["object_stem"]
    col  = REGION_TO_GT_COLUMN[row["region"]]
    if stem not in gt_indexed.index:
        return np.nan
    return gt_indexed.loc[stem, col]


meta_df = raw_meta_df.copy()
meta_df["true_class"] = meta_df.apply(lookup_true_class, axis=1)
meta_df = meta_df[meta_df["true_class"].notna()].reset_index(drop=True)

ALL_CLASSES = sorted(meta_df["true_class"].unique())
print(f"\nTarget classes ({len(ALL_CLASSES)}): {ALL_CLASSES}")
print(f"Total spectra for deployment training: {len(meta_df)}")
print("\nClass distribution:")
class_counts = meta_df["true_class"].value_counts().sort_index()
for cls, count in class_counts.items():
    print(f"  {cls:6s}: {count}")


In [ ]:
# === Cell 8: Feature Loading ===

def load_feature_vector(path):
    """Return (reflectance_vector, wavenumber_axis) for one spectrum file."""
    df = read_spectrum_file(path)
    return df["reflectance"].to_numpy(dtype=np.float64), df["wavenumber"].to_numpy()


print("Loading all spectra ...")
feature_rows, wavenumber_ref, load_errors, n_features = [], None, [], None
for _, row in meta_df.iterrows():
    try:
        vec, wn = load_feature_vector(row["path"])
        if wavenumber_ref is None:
            wavenumber_ref, n_features = wn, len(vec)
            print(f"  Reference axis: {len(wn)} points, wn=[{wn.min():.1f}, {wn.max():.1f}] cm-1")
        if len(vec) != n_features:
            raise ValueError(f"length {len(vec)} != reference {n_features}")
        feature_rows.append(vec)
    except Exception as exc:
        load_errors.append({"filename": row["filename"], "error": str(exc)})

if load_errors:
    print(f"\nWARNING: {len(load_errors)} files failed to load:")
    for e in load_errors:
        print(f"  - {e['filename']}: {e['error']}")
if not feature_rows:
    raise RuntimeError("No spectra could be loaded.")

X_all = np.vstack(feature_rows)
y_all = meta_df["true_class"].to_numpy()

print(f"\nFeature matrix: {X_all.shape}  ({X_all.shape[0]} spectra x {X_all.shape[1]} points)")
print(f"Unique objects: {meta_df['group_id'].nunique()}")


In [ ]:
# === Cell 9: Frozen HPO Hyperparameters ===

hpo_df = pd.read_csv(HPO_RESULTS_CSV)
print("HPO results:")
display(hpo_df)

# Row of the selected classifier
lr_row = hpo_df[hpo_df["classifier"] == SELECTED_CLASSIFIER]
if lr_row.empty:
    raise RuntimeError(
        f"Classifier '{SELECTED_CLASSIFIER}' not found in HPO results. "
        f"Available: {hpo_df['classifier'].tolist()}"
    )
lr_row = lr_row.iloc[0]

# best_params is stored as the repr of a dict
BEST_PARAMS = ast.literal_eval(lr_row["best_params"])

print(f"\nSelected classifier: {SELECTED_CLASSIFIER}")
print(f"Best CV macro-F1:    {lr_row['best_cv_macro_f1']:.4f}")
print(f"Best n_components:   {lr_row['best_n_components']}")
print(f"Frozen hyperparameters:")
for k, v in sorted(BEST_PARAMS.items()):
    print(f"  {k}: {v}")


In [ ]:
# === Cell 10: Deployment Fit and Metrics ===

# Benchmark pipeline StandardScaler(with_std=False) -> PCA -> SVC,
# configured with the frozen HPO hyperparameters.
# Base SVC args (probability=True, max_iter=10_000) match the benchmark grid.

deployment_pipe = Pipeline([
    ("scaler", StandardScaler(with_std=False)),
    ("pca",    PCA(random_state=RANDOM_SEED)),
    ("clf",    SVC(probability=True, random_state=RANDOM_SEED, max_iter=10_000)),
])

deployment_pipe.set_params(**BEST_PARAMS)

print("Pipeline steps:")
for name, step in deployment_pipe.named_steps.items():
    print(f"  {name}: {step}")

# --- Single fit on all data ---
print(f"\nFitting on {X_all.shape[0]} spectra ...")
deployment_pipe.fit(X_all, y_all)
print("Fit complete.")

# --- Training-set sanity check ---
y_pred_train = deployment_pipe.predict(X_all)
train_acc    = accuracy_score(y_all, y_pred_train)
train_f1     = f1_score(y_all, y_pred_train, average="macro", zero_division=0)

print(f"\nTraining accuracy:  {train_acc:.4f}")
print(f"Training macro-F1:  {train_f1:.4f}")

if train_acc < 1.0:
    print("\nWARNING: training accuracy is below 1.0 — check for configuration errors.")
    print("\nPer-class breakdown:")
    print(classification_report(y_all, y_pred_train, digits=4))
else:
    print("\nTraining sanity check PASS — 100% accuracy on all training spectra.")

print(f"\nPCA components retained: {deployment_pipe.named_steps['pca'].n_components_}")
print(f"PCA explained variance:  {deployment_pipe.named_steps['pca'].explained_variance_ratio_.sum():.4f}")

# --- Probabilistic metrics on the training set ---
# Deployment-tier (all-data) fit: these metrics are resubstitution values.
if hasattr(deployment_pipe, "predict_proba"):
    y_proba_train = deployment_pipe.predict_proba(X_all)
    # Align probability columns to ALL_CLASSES
    proba_aligned = np.zeros((len(y_all), len(ALL_CLASSES)))
    for j, cls in enumerate(deployment_pipe.classes_):
        proba_aligned[:, ALL_CLASSES.index(cls)] = y_proba_train[:, j]

    y_ohe_train = label_binarize(y_all, classes=ALL_CLASSES)
    # Multiclass Brier Score: summed squared error of one-hot truth vs. probabilities.
    train_brier = float(np.mean(np.sum((y_ohe_train - proba_aligned) ** 2, axis=1)))
    try:
        train_auc = float(roc_auc_score(
            y_ohe_train, proba_aligned, multi_class="ovr", average="macro",
        ))
    except ValueError:
        train_auc = float("nan")
else:
    train_brier = float("nan")
    train_auc   = float("nan")

print(f"\nTraining Brier Score (resubstitution): {train_brier:.4f}  (lower is better)")
print(f"Training AUC-ROC     (resubstitution): {train_auc:.4f}  (macro OVR)")


In [ ]:
# === Cell 11: Probabilistic Metrics Plot ===

# Bar charts of the resubstitution Brier Score and AUC-ROC of the deployment model.

import matplotlib.pyplot as plt

PROBA_METRICS_PNG = RESULTS_DIR / f"{PROJECT_NAME}_train_proba_metrics.png"

fig_pm, axes_pm = plt.subplots(1, 2, figsize=(10, 4.5))

# Brier Score (lower is better)
axes_pm[0].bar([SELECTED_CLASSIFIER], [train_brier],
               color="tab:red", alpha=0.85, width=0.4)
axes_pm[0].set_ylabel("Brier Score (lower is better)")
axes_pm[0].set_title("Training Brier Score (resubstitution)")
axes_pm[0].grid(axis="y", alpha=0.3)
axes_pm[0].text(0, train_brier, f"{train_brier:.4f}",
                ha="center", va="bottom", fontsize=10)

# AUC-ROC (higher is better)
axes_pm[1].bar([SELECTED_CLASSIFIER], [train_auc],
               color="tab:purple", alpha=0.85, width=0.4)
axes_pm[1].set_ylim(0, 1.05)
axes_pm[1].set_ylabel("AUC-ROC (macro OVR)")
axes_pm[1].set_title("Training AUC-ROC (resubstitution)")
axes_pm[1].grid(axis="y", alpha=0.3)
axes_pm[1].text(0, train_auc, f"{train_auc:.4f}",
                ha="center", va="bottom", fontsize=10)

fig_pm.suptitle(
    f"{SELECTED_CLASSIFIER} deployment model — probabilistic metrics "
    f"(training set, {X_all.shape[0]} spectra)",
    fontsize=12,
)
plt.tight_layout()
fig_pm.savefig(PROBA_METRICS_PNG, bbox_inches="tight", dpi=150)
plt.show()
print("Saved:", PROBA_METRICS_PNG)


In [ ]:
# === Cell 12: PCA Decision Boundary Plots ===

# Projects all spectra to 2 and 3 PCs (mean-centring only) and refits a clone of the
# tuned classifier there, for visualisation only; the full-dimensional deployment_pipe
# is the saved model.

import matplotlib.pyplot as plt
from sklearn.base import clone
from matplotlib.colors import ListedColormap, BoundaryNorm

# --- Style config (10 pt at 16 cm LaTeX text width) ---
_TEXT_WIDTH_CM = 16
_FW            = _TEXT_WIDTH_CM / 2.54          # full text width in inches
_FONT_SIZE     = 10

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _FONT_SIZE
plt.rcParams["axes.titlesize"]  = _FONT_SIZE + 1
plt.rcParams["axes.labelsize"]  = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["savefig.dpi"]     = 150

DECISION_BOUNDARY_PNG     = RESULTS_DIR / f"{PROJECT_NAME}_decision_boundary.png"
DECISION_BOUNDARY_3D_PNG  = RESULTS_DIR / f"{PROJECT_NAME}_decision_boundary_3d.png"
DECISION_BOUNDARY_3D_HTML = RESULTS_DIR / f"{PROJECT_NAME}_decision_boundary_3d.html"

# --- Project to 2D PCA ---
scaler_2d = StandardScaler(with_std=False).fit(X_all)
pca_2d    = PCA(n_components=2, random_state=RANDOM_SEED)
X_all_2d  = pca_2d.fit_transform(scaler_2d.transform(X_all))

ev = pca_2d.explained_variance_ratio_
print(f"Decision boundary visualisation -- 2D PCA projection")
print(f"  Explained variance: PC1={ev[0]:.3f}, PC2={ev[1]:.3f}, total={ev.sum():.3f}")

# --- Colour setup (one colour per class) ---
n_classes    = len(ALL_CLASSES)
class_colors = [plt.cm.tab10(i) for i in range(n_classes)]
cm_bg        = ListedColormap(class_colors)
cm_bright    = ListedColormap(class_colors)

label_to_int = {c: i for i, c in enumerate(ALL_CLASSES)}
y_all_int    = np.array([label_to_int[c] for c in y_all])

# --- Clone the tuned classifier (same hyperparameters) and refit on 2D ---
clf_2d = clone(deployment_pipe.named_steps["clf"])
clf_2d.fit(X_all_2d, y_all)

# --- Decision regions (filled contours via manual meshgrid) ---
x_min, x_max = X_all_2d[:, 0].min() - 0.5, X_all_2d[:, 0].max() + 0.5
y_min, y_max = X_all_2d[:, 1].min() - 0.5, X_all_2d[:, 1].max() + 0.5
_res = 300
_xx, _yy = np.meshgrid(np.linspace(x_min, x_max, _res),
                        np.linspace(y_min, y_max, _res))
_Z = np.array([label_to_int[c]
               for c in clf_2d.predict(np.c_[_xx.ravel(), _yy.ravel()])])
_Z = _Z.reshape(_xx.shape)
_norm = BoundaryNorm(np.arange(-0.5, n_classes, 1.0), n_classes)

fig_db, ax = plt.subplots(figsize=(8 / 2.54, 8 / 2.54))  # 8 cm square
ax.pcolormesh(_xx, _yy, _Z, cmap=cm_bg, norm=_norm, alpha=0.35, zorder=1)

ax.scatter(X_all_2d[:, 0], X_all_2d[:, 1],
           c=y_all_int, cmap=cm_bright,
           edgecolors="k", s=35, linewidths=0.5,
           vmin=0, vmax=n_classes - 1, zorder=3)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

score_2d = f1_score(y_all, clf_2d.predict(X_all_2d), average="macro", zero_division=0)
ax.set_title(f"{SELECTED_CLASSIFIER} 2D F1 = {score_2d:.3f}")
ax.set_xlabel(f"PC 1 ({ev[0]:.1%} var)")
ax.set_ylabel(f"PC 2 ({ev[1]:.1%} var)")
ax.set_box_aspect(1)

legend_handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=class_colors[i], markersize=6,
               markeredgecolor="k", markeredgewidth=0.5,
               label=ALL_CLASSES[i])
    for i in range(n_classes)
]
ax.legend(handles=legend_handles, loc="best", framealpha=0.9,
          borderpad=0.3, labelspacing=0.2, handletextpad=0.4, fontsize=_FONT_SIZE - 2)


plt.tight_layout()
fig_db.savefig(DECISION_BOUNDARY_PNG, bbox_inches="tight", dpi=150)
plt.show()
print("Saved:", DECISION_BOUNDARY_PNG)

# --- 3D PCA projection ---
# Points are coloured by true class; a classifier refit on 3 PCs gives a 3D macro-F1.
# Outputs a static PNG and, if plotly is installed, an interactive HTML figure.

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3d proj)

# --- Project to 3D PCA (same mean-centre-only scaler convention) ---
pca_3d   = PCA(n_components=3, random_state=RANDOM_SEED)
X_all_3d = pca_3d.fit_transform(scaler_2d.transform(X_all))
ev3      = pca_3d.explained_variance_ratio_
print(f"\n3D PCA projection — explained variance: "
      f"PC1={ev3[0]:.3f}, PC2={ev3[1]:.3f}, PC3={ev3[2]:.3f}, "
      f"total={ev3.sum():.3f}")

# --- Clone tuned classifier and refit on 3D ---
clf_3d = clone(deployment_pipe.named_steps["clf"])
clf_3d.fit(X_all_3d, y_all)
score_3d = f1_score(y_all, clf_3d.predict(X_all_3d),
                    average="macro", zero_division=0)
print(f"3D-refit training macro-F1: {score_3d:.3f} "
      f"(2D was {score_2d:.3f}, full-dim {train_f1:.3f})")

# --- Static matplotlib 3D scatter ---
fig_db3 = plt.figure(figsize=(_FW, _FW))   # square at full text width
ax3 = fig_db3.add_subplot(111, projection="3d")
for i, cls in enumerate(ALL_CLASSES):
    mask = (y_all == cls)
    ax3.scatter(X_all_3d[mask, 0], X_all_3d[mask, 1], X_all_3d[mask, 2],
                color=class_colors[i], label=cls,
                edgecolors="k", linewidths=0.3, s=30, alpha=0.85)
ax3.set_xlabel(f"PC 1 ({ev3[0]:.1%})")
ax3.set_ylabel(f"PC 2 ({ev3[1]:.1%})")
ax3.set_zlabel(f"PC 3 ({ev3[2]:.1%})")
ax3.set_title(
    f"{SELECTED_CLASSIFIER} Deployment Model — 3D PCA Projection\n"
    f"3D F1 = {score_3d:.3f} | Full-Dim Train F1 = {train_f1:.3f} "
    f"(3D var = {ev3.sum():.1%})",
)
ax3.legend(loc="upper left", fontsize=_FONT_SIZE - 2, framealpha=0.9)
plt.tight_layout()
fig_db3.savefig(DECISION_BOUNDARY_3D_PNG, bbox_inches="tight", dpi=150)
plt.show()
print("Saved:", DECISION_BOUNDARY_3D_PNG)

# --- Interactive Plotly 3D scatter (rotatable HTML) ---
try:
    import plotly.graph_objects as go

    fig_plotly = go.Figure()
    for i, cls in enumerate(ALL_CLASSES):
        mask = (y_all == cls)
        rgba = tuple(int(255 * c) for c in class_colors[i][:3])
        fig_plotly.add_trace(go.Scatter3d(
            x=X_all_3d[mask, 0], y=X_all_3d[mask, 1], z=X_all_3d[mask, 2],
            mode="markers", name=cls,
            marker=dict(size=4,
                        color=f"rgb{rgba}",
                        line=dict(width=0.5, color="black")),
            text=meta_df["filename"].to_numpy()[mask],
            hovertemplate="%{text}<br>"
                          "PC1=%{x:.2f}<br>PC2=%{y:.2f}<br>PC3=%{z:.2f}"
                          "<extra>" + cls + "</extra>",
        ))
    fig_plotly.update_layout(
        title=(f"{SELECTED_CLASSIFIER} Deployment Model — 3D PCA Projection "
               f"(Total Var {ev3.sum():.1%})"),
        scene=dict(
            xaxis_title=f"PC 1 ({ev3[0]:.1%})",
            yaxis_title=f"PC 2 ({ev3[1]:.1%})",
            zaxis_title=f"PC 3 ({ev3[2]:.1%})",
        ),
        legend=dict(itemsizing="constant"),
        width=900, height=750,
    )
    fig_plotly.write_html(str(DECISION_BOUNDARY_3D_HTML))
    print("Saved:", DECISION_BOUNDARY_3D_HTML)
    try:
        fig_plotly.show()
    except Exception:
        pass  # headless / non-notebook context
except ImportError:
    print("plotly not available — skipped interactive 3D HTML export. "
          "Install with: pip install plotly")

In [ ]:
# === Cell 13: Save Deployment Model ===

# Save the fitted pipeline as .joblib for the pipeline demonstrator.
joblib.dump(deployment_pipe, DEPLOYMENT_MODEL_JOBLIB)
model_size_kb = DEPLOYMENT_MODEL_JOBLIB.stat().st_size / 1024

print(f"Deployment model saved to: {DEPLOYMENT_MODEL_JOBLIB}")
print(f"Model size: {model_size_kb:.1f} KB")

# Training summary CSV
summary = pd.DataFrame([{
    "project":           PROJECT_NAME,
    "classifier":        SELECTED_CLASSIFIER,
    "hpo_source":        str(HPO_RESULTS_CSV),
    "spectra_dir":       str(SPECTRA_DIR),
    "n_training_spectra": X_all.shape[0],
    "n_features":        X_all.shape[1],
    "n_classes":         len(ALL_CLASSES),
    "classes":           str(ALL_CLASSES),
    "pca_n_components":  int(deployment_pipe.named_steps["pca"].n_components_),
    "best_params":       str(BEST_PARAMS),
    "train_accuracy":    train_acc,
    "train_macro_f1":    train_f1,
    "train_brier_score": train_brier,
    "train_auc_roc":     train_auc,
    "model_size_kb":     round(model_size_kb, 1),
    "model_path":        str(DEPLOYMENT_MODEL_JOBLIB),
    "sklearn_version":   sklearn.__version__,
}])
summary.to_csv(TRAINING_SUMMARY_CSV, index=False)
print(f"Training summary saved to: {TRAINING_SUMMARY_CSV}")

print("\n" + "=" * 70)
print("Deployment training complete.")
print(f"To use this model in the pipeline demonstrator, set:")
print(f"  IR_CLASSIFIER_MODEL_PATH = Path(\"{DEPLOYMENT_MODEL_JOBLIB}\")")
print("=" * 70)
